## 0. Setup & dependencies

In [1]:
!pip install -q chromadb
!pip install -q ollama
!pip install -q rank_bm25
!pip install -q \
  sentence-transformers==3.3.1 \
  transformers==4.47.1 \
  tokenizers==0.21.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

## Pulling Gemma4 model

In [2]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version
!ollama serve > ollama.log 2>&1 &
!sleep 5
!ollama pull gemma4

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 71 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
ollama version is 0.32.14



In [3]:
import json
import re
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np

from enum import Enum
from pydantic import BaseModel, Field


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Ingestion — load & validate

In [5]:
def load_and_validate(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    if not path.is_file():
        raise ValueError(f"Not a file: {path}")
    text = path.read_text(encoding="utf-8")
    if not text.strip():
        raise ValueError("Input file is empty")
    return text


# Adjust path for your environment (Colab Drive, local, or artifacts)
DATA_CANDIDATES = [
    Path("/content/drive/MyDrive/civil-code.txt"),
    Path("data/civil-code.txt"),
    Path("/home/workdir/attachments/02_civil-code_قانون-مدنی_1314-08-08_qavanin_12021850837713548188.txt"),
]

raw_text = None
for p in DATA_CANDIDATES:
    if p.exists():
        raw_text = load_and_validate(p)
        print(f"Loaded {len(raw_text):,} chars from {p}")
        break

if raw_text is None:
    raise FileNotFoundError("Civil Code TXT not found in any candidate path")

Loaded 228,132 chars from /content/drive/MyDrive/civil-code.txt


In [6]:
def split_metadata_body(text: str) -> Tuple[dict, str]:
    lines = text.splitlines()
    metadata_lines = []
    body_start = 0
    in_metadata = False

    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.startswith("# ===== METADATA ====="):
            in_metadata = True
            continue
        if in_metadata and stripped.startswith("# ===================="):
            body_start = i + 1
            break
        if in_metadata:
            metadata_lines.append(line)

    if not metadata_lines:
        # Fallback: no metadata block — treat entire file as body
        return {}, text.strip()

    metadata = {}
    for line in metadata_lines:
        line = line.strip()
        if not line.startswith("#") or ":" not in line:
            continue
        key, value = line[1:].strip().split(":", 1)
        metadata[key.strip()] = value.strip()

    body = "\n".join(lines[body_start:]).strip()
    return metadata, body


doc_metadata, body = split_metadata_body(raw_text)
print("Metadata keys:", list(doc_metadata.keys()))
print("Body length:", len(body))
DOC_ID = doc_metadata.get("doc_id", "12021850837713548188")
LAW_TITLE = doc_metadata.get("title_fa", "قانون مدنی")

Metadata keys: ['title_fa', 'title_en', 'approval_date_jalali', 'source', 'source_url', 'text_url', 'doc_id', 'version', 'retrieved_at', 'encoding']
Body length: 227637


## 2. Provision extraction (articles, notes, hierarchy, amendments)

In [7]:
HEADER_FIELDS = {
    "جلد": ("book", 1),
    "قسمت": ("part", 2),
    "باب": ("chapter", 3),
    "فصل": ("section", 4),
    "مبحث": ("subsection", 5),
}
FIELD_ORDER = ["book", "part", "chapter", "section", "subsection"]

HEADER_RE = re.compile(r"^(جلد|قسمت|باب|فصل|مبحث)\b\s*(.*)$")
MADDEH_RE = re.compile(
    r"^ماده\s+([۰-۹0-9]+(?:\s*(?:مكرر|مکرر))?)"
    r"\s*(?:\(([^)]+)\))?\s*[-–ـ]?\s*(.*)$"
)
TABSARE_RE = re.compile(
    r"^تبصره\s*([۰-۹0-9]+|[\u0600-\u06FF]+)?"
    r"\s*(?:\(([^)]+)\))?\s*[-–ـ]?\s*(.*)$"
)


def parse_amendment(raw: Optional[str]) -> Tuple[Optional[str], Optional[str]]:
    if not raw:
        return None, None
    parts = raw.strip().split(None, 1)
    status = parts[0] if parts else None
    date_raw = parts[1] if len(parts) > 1 else None
    date_norm = None
    if date_raw:
        nums = re.findall(r"\d+", date_raw)
        if len(nums) == 3:
            dd, mm, yyyy = nums
            date_norm = f"{yyyy}/{mm.zfill(2)}/{dd.zfill(2)}"
    return status, date_norm


def extract_provisions(body: str, doc_id: str, law_title: str) -> List[dict]:
    hierarchy = {f: None for f in FIELD_ORDER}
    provisions = []
    current = None
    last_article_number = None

    def flush():
        nonlocal current
        if current is None:
            return
        text = " ".join(t.strip() for t in current["text_lines"] if t.strip())
        status, date = parse_amendment(current["amendment_raw"])
        provisions.append({
            "doc_id": doc_id,
            "law_title": law_title,
            "article_number": current["article_number"],
            "provision_type": current["type"],
            "text": text,
            "parent_article_number": current.get("parent_article_number"),
            "book": current["hierarchy"]["book"],
            "part": current["hierarchy"]["part"],
            "chapter": current["hierarchy"]["chapter"],
            "section": current["hierarchy"]["section"],
            "subsection": current["hierarchy"]["subsection"],
            "amendment_status": status,
            "amendment_date_jalali": date,
        })
        current = None

    for raw_line in body.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        m = HEADER_RE.match(line)
        if m:
            flush()
            marker = m.group(1)
            field, rank = HEADER_FIELDS[marker]
            hierarchy[field] = line
            for deeper in FIELD_ORDER[rank:]:
                hierarchy[deeper] = None
            continue

        m = MADDEH_RE.match(line)
        if m:
            flush()
            article_number, amendment_raw, rest = m.groups()
            last_article_number = article_number
            current = {
                "type": "article",
                "article_number": article_number,
                "amendment_raw": amendment_raw,
                "text_lines": [rest] if rest else [],
                "hierarchy": dict(hierarchy),
            }
            continue

        m = TABSARE_RE.match(line)
        if m:
            flush()
            label, amendment_raw, rest = m.groups()
            if last_article_number is None:
                continue
            article_number = f"{last_article_number} تبصره"
            if label:
                article_number += f" {label}"
            current = {
                "type": "note",
                "article_number": article_number,
                "amendment_raw": amendment_raw,
                "text_lines": [rest] if rest else [],
                "hierarchy": dict(hierarchy),
                "parent_article_number": last_article_number,
            }
            continue

        if current is not None:
            current["text_lines"].append(line)

    flush()
    return provisions


provisions = extract_provisions(body, DOC_ID, LAW_TITLE)
print(f"Extracted {len(provisions)} provisions")
print("Sample amendment statuses:",
      set(p["amendment_status"] for p in provisions if p["amendment_status"]))

Extracted 1352 provisions
Sample amendment statuses: {'اصلاحي', 'الحاقي', 'منسوخه', 'جایگزین'}


## 3. Persian normalization

In [8]:
DIGIT_MAP = str.maketrans({
    "۰": "0", "۱": "1", "۲": "2", "۳": "3", "۴": "4",
    "۵": "5", "۶": "6", "۷": "7", "۸": "8", "۹": "9",
    "٠": "0", "١": "1", "٢": "2", "٣": "3", "٤": "4",
    "٥": "5", "٦": "6", "٧": "7", "٨": "8", "٩": "9",
})
LETTER_MAP = str.maketrans({"ك": "ک", "ي": "ی", "ى": "ی"})

DIACRITICS_RE = re.compile(
    "[\u0610-\u061a\u064b-\u065f\u06d6-\u06dc\u06df-\u06e8\u06ea-\u06ed\u0670]"
)
TATWEEL = "\u0640"
ZWNJ = "\u200c"
REPEATED_ZWNJ_RE = re.compile(r"\u200c+")
ZWNJ_ADJACENT_SPACE_RE = re.compile(r" ?\u200c ?")
WHITESPACE_RE = re.compile(r" {2,}")
BOM = "\ufeff"
ZWSP = "\u200b"
NBSP = "\u00a0"
CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]")
ALL_WHITESPACE_RE = re.compile(r"\s+")


def clean(text: str) -> str:
    if not text:
        return ""
    text = text.replace(BOM, "").replace(ZWSP, "")
    text = text.replace(NBSP, " ")
    text = text.replace("\n", " ").replace("\t", " ").replace("\r", " ")
    text = CONTROL_RE.sub("", text)
    text = ALL_WHITESPACE_RE.sub(" ", text)
    return text.strip()


def normalize(text: str) -> str:
    if not text:
        return ""
    text = text.translate(DIGIT_MAP)
    text = text.translate(LETTER_MAP)
    text = text.replace(TATWEEL, "")
    text = DIACRITICS_RE.sub("", text)
    text = REPEATED_ZWNJ_RE.sub(ZWNJ, text)
    text = ZWNJ_ADJACENT_SPACE_RE.sub(
        lambda m: " " if m.group(0) != ZWNJ else ZWNJ, text
    )
    text = WHITESPACE_RE.sub(" ", text)
    return text.strip()


def normalize_article_number(s: str) -> str:
    if not s:
        return s
    s = s.translate(DIGIT_MAP).translate(LETTER_MAP)
    return re.sub(r"\s+", " ", s).strip()


def enrich_provisions(provisions: List[dict]) -> List[dict]:
    for p in provisions:
        p["text_normalized"] = normalize(clean(p.get("text") or ""))
        p["article_number_normalized"] = normalize_article_number(
            p.get("article_number") or ""
        )
        parent = p.get("parent_article_number")
        p["parent_article_number_normalized"] = (
            normalize_article_number(parent) if parent else None
        )
    return provisions


provisions = enrich_provisions(provisions)
print("Normalization sample:")
print("  raw  :", provisions[0]["text"][:80])
print("  norm :", provisions[0]["text_normalized"][:80])

Normalization sample:
  raw  : مصوبات مجلس شوراي اسلامي و نتيجه همه پرسي پس از طي مراحل قانوني به رئيس جمهور اب
  norm : مصوبات مجلس شورای اسلامی و نتیجه همه پرسی پس از طی مراحل قانونی به رئیس جمهور اب


## 4. Chunking — one chunk per article (notes merged)

In [9]:
def build_chunks(provisions: List[dict]) -> List[dict]:
    """One chunk per article; تبصره notes are merged into their parent."""
    notes_by_parent = defaultdict(list)
    for p in provisions:
        if p["provision_type"] == "note":
            key = (p["doc_id"], p.get("parent_article_number_normalized"))
            notes_by_parent[key].append(p)

    chunks = []
    for p in provisions:
        if p["provision_type"] == "note":
            continue

        notes = notes_by_parent.get(
            (p["doc_id"], p["article_number_normalized"]), []
        )

        text = p["text"]
        text_normalized = p.get("text_normalized") or ""
        note_numbers = []

        for note in notes:
            text += f" [{note['article_number']}] {note['text']}"
            text_normalized += (
                f" [{note['article_number_normalized']}] "
                f"{note.get('text_normalized') or ''}"
            )
            note_numbers.append(note["article_number"])

        chunks.append({
            "chunk_id": f"{p['doc_id']}:{p['article_number_normalized']}",
            "doc_id": p["doc_id"],
            "law_title": p["law_title"],
            "article_number": p["article_number"],
            "article_number_normalized": p["article_number_normalized"],
            "provision_type": p["provision_type"],
            "text": text.strip(),
            "text_normalized": text_normalized.strip(),
            "note_article_numbers": note_numbers,
            "book": p.get("book"),
            "part": p.get("part"),
            "chapter": p.get("chapter"),
            "section": p.get("section"),
            "subsection": p.get("subsection"),
            "amendment_status": p.get("amendment_status"),
            "amendment_date_jalali": p.get("amendment_date_jalali"),
        })

    return chunks


chunks = build_chunks(provisions)
print(f"Total chunks: {len(chunks)}")
print(f"Chunks with notes: {sum(1 for c in chunks if c['note_article_numbers'])}")
print(f"Abrogated / amended samples:")
for c in chunks:
    if c["amendment_status"]:
        print(f"  ماده {c['article_number']} [{c['amendment_status']}]")
        if sum(1 for x in chunks if x['amendment_status']) > 5:
            break

Total chunks: 1330
Chunks with notes: 18
Abrogated / amended samples:
  ماده 1 [اصلاحي]


## 5. Dense index (Jina embeddings-v3 + Chroma)

In [10]:
from sentence_transformers import SentenceTransformer
import chromadb

embed_model = SentenceTransformer(
    "jinaai/jina-embeddings-v3",
    trust_remote_code=True,
)

texts = [c["text_normalized"] for c in chunks]
embeddings = embed_model.encode(
    texts,
    task="retrieval.passage",
    batch_size=32,
    show_progress_bar=True,
)
print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

custom_st.py: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/192 [00:00<?, ?B/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Embedding shape: (1330, 1024)


In [11]:
client = chromadb.PersistentClient(path="data/processed/vector_store")

# Recreate cleanly for reproducibility
try:
    client.delete_collection("civil_code")
except Exception:
    pass

collection = client.create_collection(
    name="civil_code",
    metadata={"hnsw:space": "cosine"},
)


def clean_meta(c):
    return {
        "doc_id": str(c["doc_id"]),
        "law_title": c["law_title"] or "",
        "article_number": c["article_number"] or "",
        "article_number_normalized": c["article_number_normalized"] or "",
        "provision_type": c["provision_type"] or "",
        "book": c["book"] or "",
        "part": c["part"] or "",
        "chapter": c["chapter"] or "",
        "section": c["section"] or "",
        "subsection": c["subsection"] or "",
        "amendment_status": c["amendment_status"] or "",
        "amendment_date_jalali": c["amendment_date_jalali"] or "",
    }


collection.add(
    ids=[c["chunk_id"] for c in chunks],
    embeddings=embeddings.tolist(),
    documents=[c["text"] for c in chunks],
    metadatas=[clean_meta(c) for c in chunks],
)
print(collection.count(), "vectors stored")

chunk_by_id = {c["chunk_id"]: c for c in chunks}

1330 vectors stored


## 6. BM25 + Hybrid search (RRF)

In [12]:
from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r"[\u0600-\u06FF]+|\d+")


def tokenize(text: str) -> List[str]:
    return TOKEN_RE.findall(text)


tokenized_corpus = [tokenize(c["text_normalized"]) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)


def bm25_search(query: str, top_k: int = 20) -> List[str]:
    q = normalize(clean(query))
    scores = bm25.get_scores(tokenize(q))
    ranked = sorted(range(len(scores)), key=lambda i: -scores[i])[:top_k]
    return [chunks[i]["chunk_id"] for i in ranked]


def vector_search(query: str, top_k: int = 20) -> List[str]:
    q = normalize(clean(query))
    query_vec = embed_model.encode([q], task="retrieval.query")[0]
    result = collection.query(
        query_embeddings=[query_vec.tolist()], n_results=top_k
    )
    return result["ids"][0]


def reciprocal_rank_fusion(ranked_lists, k: int = 60):
    scores = {}
    for ranked in ranked_lists:
        for rank, chunk_id in enumerate(ranked, start=1):
            scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])


def hybrid_search(query: str, top_k: int = 10, candidate_k: int = 30):
    bm25_ranked = bm25_search(query, top_k=candidate_k)
    vector_ranked = vector_search(query, top_k=candidate_k)
    fused = reciprocal_rank_fusion([bm25_ranked, vector_ranked])[:top_k]
    return [(chunk_by_id[cid], score) for cid, score in fused]


# Smoke test
print("Hybrid smoke test — مال غیرمنقول:")
for chunk, score in hybrid_search("مال غیرمنقول", top_k=5):
    print(f"  {chunk['article_number']:>8}  {score:.4f}  {chunk['text'][:60]}...")

Hybrid smoke test — مال غیرمنقول:
        12  0.0328  مال غيرمنقول آنست كه از محلي بمحل ديگر نتوان نقل نمود اعم از...
        11  0.0315  اموال بر دو قسم است منقول و غيرمنقول....
         8  0.0313  اموال غيرمنقول كه اتباع خارجه در ايران بر طبق عهود تملك كرده...
        58  0.0305  فقط وقف مالي جائز است كه با بقاي عين بتوان از آن منتفع شد اع...
        18  0.0300  حق انتفاع از اشياء غيرمنقوله مثل حق عمري و سكني و همچنين حق ...


## 6b. Deterministic query analysis

Explicit legal identifiers (`ماده ۳۰`, `قانون مدنی`) are **hard constraints**.  
They are extracted with regex + a known-law list — not an LLM — so retrieval precision does not depend on generative guessing.

Outputs feed metadata filters / boosts before RRF and reranking.


In [22]:
"""
Query analysis for the legal RAG pipeline.

- Normalize Persian/Arabic digits and common character variants
- Detect explicit article references (ماده 30, ماده ۲۱۸ مکرر, ...)
- Detect known law titles
- Classify retrieval intent (article_lookup / legal_question / multi_issue)
- Does NOT answer and does NOT retrieve
"""

class QueryIntent(str, Enum):
    ARTICLE_LOOKUP = "article_lookup"
    LEGAL_QUESTION = "legal_question"
    MULTI_ISSUE = "multi_issue"
    UNKNOWN = "unknown"


class QueryAnalysis(BaseModel):
    original_query: str
    normalized_query: str
    intent: QueryIntent
    law_title: Optional[str] = None
    article_number: Optional[str] = None
    article_number_normalized: Optional[str] = None
    keywords: List[str] = Field(default_factory=list)
    has_explicit_article: bool = False
    has_explicit_law: bool = False


class QueryAnalyzer:
    """Deterministic analyzer for Persian legal queries."""

    _ARTICLE_RE = re.compile(
        r"""
        \b
        ماده
        \s+
        (?P<number>\d+)
        (?:
            \s*
            (?P<suffix>مکرر|مكرر)
        )?
        \b
        """,
        re.IGNORECASE | re.VERBOSE,
    )

    _LAW_RE = re.compile(
        r"""
        قانون
        \s+
        (?P<title>
            [\u0600-\u06FF]+
            (?:
                \s+
                [\u0600-\u06FF]+
            ){0,5}
        )
        """,
        re.VERBOSE,
    )

    _STOPWORDS = {
        "چیست", "چیست؟", "چه", "چگونه", "چطور", "آیا",
        "است", "هست", "هستند", "باشد", "میشود", "می‌شود", "شود",
        "دارد", "دارند", "درباره", "در", "به", "از", "برای", "با",
        "که", "و", "یا", "را", "این", "آن", "یک",
        "طبق", "مطابق", "بر اساس", "قانون", "ماده",
    }

    def __init__(self, known_laws: Optional[List[str]] = None):
        self.known_laws = [
            self._normalize_basic(law) for law in (known_laws or [])
        ]

    def analyze(self, query: str) -> QueryAnalysis:
        if query is None or not str(query).strip():
            raise ValueError("Query cannot be empty.")

        original_query = query
        normalized_query = self._normalize_query(query)
        article_number = self._extract_article_number(normalized_query)
        law_title = self._extract_law_title(normalized_query)
        keywords = self._extract_keywords(
            normalized_query,
            article_number=article_number,
            law_title=law_title,
        )
        intent = self._classify_intent(
            normalized_query,
            article_number=article_number,
            law_title=law_title,
        )
        return QueryAnalysis(
            original_query=original_query,
            normalized_query=normalized_query,
            intent=intent,
            law_title=law_title,
            article_number=article_number,
            article_number_normalized=(
                self._normalize_article_number(article_number)
                if article_number
                else None
            ),
            keywords=keywords,
            has_explicit_article=article_number is not None,
            has_explicit_law=law_title is not None,
        )

    def _normalize_query(self, query: str) -> str:
        # Reuse project clean + normalize, then force digits/letters for regex
        text = normalize(clean(query))
        text = text.translate(DIGIT_MAP)
        text = text.translate(LETTER_MAP)
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    @staticmethod
    def _normalize_basic(text: str) -> str:
        text = text.translate(DIGIT_MAP).translate(LETTER_MAP)
        return re.sub(r"\s+", " ", text).strip()

    def _extract_article_number(self, query: str) -> Optional[str]:
        match = self._ARTICLE_RE.search(query)
        if not match:
            return None
        number = match.group("number")
        suffix = match.group("suffix")
        if suffix:
            return f"{number} مکرر"
        return number

    @staticmethod
    def _normalize_article_number(article_number: str) -> str:
        if not article_number:
            return article_number
        article_number = article_number.translate(DIGIT_MAP).translate(LETTER_MAP)
        article_number = re.sub(r"\s+", " ", article_number).strip()
        article_number = re.sub(r"\s*(مکرر|مكرر)$", " مکرر", article_number)
        return article_number

    def _extract_law_title(self, query: str) -> Optional[str]:
        for law in sorted(self.known_laws, key=len, reverse=True):
            if law in query:
                return law
        match = self._LAW_RE.search(query)
        if not match:
            return None
        title = match.group("title").strip()
        if not title:
            return None
        return f"قانون {title}"

    def _extract_keywords(
        self,
        query: str,
        article_number: Optional[str],
        law_title: Optional[str],
    ) -> List[str]:
        text = query
        if article_number:
            escaped = re.escape(article_number)
            text = re.sub(rf"\bماده\s+{escaped}\b", " ", text)
        if law_title:
            text = text.replace(law_title, " ")
        tokens = re.findall(r"[\w\u0600-\u06FF]+", text, flags=re.UNICODE)
        keywords = []
        for token in tokens:
            if not token.strip() or token in self._STOPWORDS or token.isdigit():
                continue
            if token not in keywords:
                keywords.append(token)
        return keywords

    def _classify_intent(
        self,
        query: str,
        article_number: Optional[str],
        law_title: Optional[str],
    ) -> QueryIntent:
        if self._looks_multi_issue(query):
            return QueryIntent.MULTI_ISSUE
        if article_number is not None:
            return QueryIntent.ARTICLE_LOOKUP
        if law_title is not None or self._looks_like_question(query):
            return QueryIntent.LEGAL_QUESTION
        return QueryIntent.UNKNOWN

    @staticmethod
    def _looks_multi_issue(query: str) -> bool:
        separators = [" و همچنین ", " و نیز ", " همچنین ", "؛", ";"]
        if sum(query.count(s) for s in separators) >= 1:
            return True
        q_markers = ["چیست", "چگونه", "چطور", "آیا", "چه"]
        hit = sum(1 for m in q_markers if m in query)
        if hit >= 2 and " و " in query:
            return True
        return False

    @staticmethod
    def _looks_like_question(query: str) -> bool:
        markers = ("?", "؟", "چیست", "چگونه", "چطور", "آیا", "چه شرایطی", "چه حکمی", "چه مواردی")
        return any(m in query for m in markers)


# Known laws in this prototype (extend when multi-law corpus is available)
KNOWN_LAWS = list({globals().get("LAW_TITLE") or "قانون مدنی", "قانون مدنی"})
query_analyzer = QueryAnalyzer(known_laws=KNOWN_LAWS)

# Quick sanity checks
for q in [
    "ماده ۳۰ قانون مدنی چه می‌گوید؟",
    "شرایط شاهد در دادگاه چیست؟",
    "مال غیرمنقول چیست و همچنین حق انتفاع چگونه تعریف می‌شود؟",
    "ماده ۲۱۸ مکرر",
]:
    a = query_analyzer.analyze(q)
    print(f"Q: {q}")
    print(f"  intent={a.intent.value}  article={a.article_number_normalized}  law={a.law_title}  kw={a.keywords[:5]}")
    print()


Q: ماده ۳۰ قانون مدنی چه می‌گوید؟
  intent=article_lookup  article=30  law=قانون مدنی  kw=['می', 'گوید؟']

Q: شرایط شاهد در دادگاه چیست؟
  intent=legal_question  article=None  law=None  kw=['شرایط', 'شاهد', 'دادگاه']

Q: مال غیرمنقول چیست و همچنین حق انتفاع چگونه تعریف می‌شود؟
  intent=multi_issue  article=None  law=None  kw=['مال', 'غیرمنقول', 'همچنین', 'حق', 'انتفاع']

Q: ماده ۲۱۸ مکرر
  intent=article_lookup  article=218 مکرر  law=None  kw=[]



In [24]:
def _art_key(s) -> str:
    """Canonical article key for matching."""
    if not s:
        return ""
    s = str(s).translate(DIGIT_MAP).translate(LETTER_MAP)
    s = s.replace("مكرر", "مکرر")
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s*مکرر\s*$", " مکرر", s)
    return s.strip()


def lookup_by_article(article_number_normalized: str, law_title: str | None = None) -> list:
    """Direct metadata lookup for explicit article queries."""
    target = _art_key(article_number_normalized)
    if not target:
        return []
    hits = []
    for c in chunks:
        art = _art_key(c.get("article_number_normalized") or c.get("article_number"))
        if art == target:
            hits.append(c)
    return hits


def apply_constraints(ranked: list, analysis=None, boost: float = 3.0):
    if analysis is None:
        return ranked
    scored = []
    for chunk, score in ranked:
        s = float(score)
        if analysis.article_number_normalized:
            art = _art_key(chunk.get("article_number_normalized") or chunk.get("article_number"))
            target = _art_key(analysis.article_number_normalized)
            if art == target:
                s = max(s, 1.0) * boost
            elif target and target in art:
                s = max(s, 0.5) * boost * 0.8
        if analysis.law_title and analysis.law_title in (chunk.get("law_title") or ""):
            s *= 1.2
        scored.append((chunk, s))
    scored.sort(key=lambda x: -x[1])
    return scored


def hybrid_search_analyzed(query: str, analysis=None, top_k: int = 10, candidate_k: int = 30):
    if analysis is None:
        analysis = query_analyzer.analyze(query)

    if analysis.intent == QueryIntent.MULTI_ISSUE:
        candidate_k = max(candidate_k, 40)
        top_k = max(top_k, 12)

    search_text = analysis.normalized_query or query
    bm25_ranked = bm25_search(search_text, top_k=candidate_k)
    vector_ranked = vector_search(search_text, top_k=candidate_k)
    fused = reciprocal_rank_fusion([bm25_ranked, vector_ranked])[
        : max(top_k * 2, candidate_k // 2)
    ]
    pairs = [(chunk_by_id[cid], score) for cid, score in fused]

    # Explicit article → inject metadata hits (fixes ماده ۱۰)
    if analysis.has_explicit_article and analysis.article_number_normalized:
        direct = lookup_by_article(analysis.article_number_normalized)
        seen = {c["chunk_id"] for c, _ in pairs}
        for c in direct:
            if c["chunk_id"] not in seen:
                pairs.append((c, 2.0))
                seen.add(c["chunk_id"])
            else:
                pairs = [
                    (ch, max(sc, 2.0) if ch["chunk_id"] == c["chunk_id"] else sc)
                    for ch, sc in pairs
                ]
        print(f"lookup_by_article({analysis.article_number_normalized!r}) -> {len(direct)} hit(s)")
        if not direct:
            print(
                f"[warn] no chunk for article={analysis.article_number_normalized!r}; "
                f"sample={[c.get('article_number_normalized') for c in chunks[:8]]}"
            )

    pairs = apply_constraints(pairs, analysis)
    return pairs[:top_k], analysis


# Self-check
_a = query_analyzer.analyze("ماده ۱۰ قانون مدنی درباره چیست؟")
print("analyzer:", _a.intent.value, "article=", _a.article_number_normalized)
_hits = lookup_by_article("10")
print(f"lookup_by_article('10') -> {len(_hits)} hit(s):", [h["article_number"] for h in _hits])
_pairs, _ = hybrid_search_analyzed("ماده ۱۰ قانون مدنی درباره چیست؟", analysis=_a)
print("top:", [(c["article_number_normalized"], round(s, 3)) for c, s in _pairs[:5]])

analyzer: article_lookup article= 10
lookup_by_article('10') -> 1 hit(s): ['10']
lookup_by_article('10') -> 1 hit(s)
top: [('10', 7.2), ('1022', 1.44), ('1088', 1.44), ('1104', 1.44), ('1335', 0.036)]


## 7. Cross-encoder reranker + amendment-aware scoring

In [25]:
from transformers import AutoModelForSequenceClassification
import torch

reranker_model = AutoModelForSequenceClassification.from_pretrained(
    "jinaai/jina-reranker-v2-base-multilingual",
    torch_dtype="auto",
    trust_remote_code=True,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
reranker_model.to(device)
reranker_model.eval()
print(f"Reranker on {device}")


def is_abrogated(chunk: dict) -> bool:
    status = (chunk.get("amendment_status") or "").lower()
    return "منسوخ" in status or "نسخ" in status


def rerank(query: str, candidates: list, top_k: int = 5):
    """Cross-encoder rerank; lightly down-weight abrogated articles."""
    if not candidates:
        return []

    q = normalize(clean(query))
    pairs = [[q, chunk["text_normalized"]] for chunk, _ in candidates]
    scores = reranker_model.compute_score(pairs, max_length=1024)
    if isinstance(scores, float):
        scores = [scores]

    combined = []
    for (chunk, hybrid_score), score in zip(candidates, scores):
        final = float(score)
        if is_abrogated(chunk):
            final *= 0.4  # still retrievable (for historical questions) but ranked lower
        combined.append((chunk, final, hybrid_score))

    combined.sort(key=lambda x: -x[1])
    return combined[:top_k]


# Smoke test
cands = hybrid_search("شرایط شاهد", top_k=10)
results = rerank("شرایط شاهد", cands, top_k=5)
print("Rerank smoke test:")
for chunk, score, h in results:
    flag = f" [{chunk['amendment_status']}]" if chunk.get("amendment_status") else ""
    print(f"  {chunk['article_number']:>10}{flag}  rerank={score:.3f}  {chunk['text'][:55]}...")

Reranker on cuda
Rerank smoke test:
        1313 [اصلاحي]  rerank=0.400  در شاهد بلوغ، عقل، عدالت، ايمان و طهارت مولد شرط است. [...
        1320  rerank=0.260  شهادت بر شهادت در صورتي مسموع است كه شاهد اصل وفات يافت...
        1312  rerank=0.238  احكام مذكور در فوق در موارد ذيل جاري نخواهد بود. 1 - در...
          78  rerank=0.220  واقف ميتواند بر متولي ناظر قرار دهد كه اعمال متولي بتصو...
        1319  rerank=0.203  در صورتيكه شاهد از شهادت خود رجوع كند يا معلوم شود بر خ...


## 7b. Evidence selection

Sits between reranking and generation. Decides **whether** evidence is sufficient and **which** chunks the LLM may see.

- Explicit article queries require a matching article (and optional law match).
- General questions need a minimum reranker score.
- Abstention is deterministic and logged with a machine-readable `reason`.


In [26]:
"""
Evidence selection for the legal RAG pipeline.
"""

from dataclasses import dataclass


@dataclass
class EvidenceItem:
    chunk: dict
    score: float
    rank: int
    hybrid_score: float
    chunk_id: str
    doc_id: str
    law_title: str
    article_number: str
    note_article_numbers: list


@dataclass
class EvidenceResult:
    sufficient: bool
    items: list
    reason: str
    top_score: float | None = None
    explicit_article_query: bool = False
    explicit_law_query: bool = False

    @property
    def should_abstain(self) -> bool:
        return not self.sufficient

    def as_rerank_tuples(self):
        return [(item.chunk, item.score, item.hybrid_score) for item in self.items]


class EvidenceSelector:
    def __init__(
        self,
        top_k: int = 5,
        min_score: float = 0.0,
        min_explicit_article_score: float | None = None,
        require_article_match: bool = True,
    ):
        self.top_k = top_k
        self.min_score = min_score
        self.min_explicit_article_score = (
            min_explicit_article_score
            if min_explicit_article_score is not None
            else min_score
        )
        self.require_article_match = require_article_match
        self.abs_floor = 0.25

    def select(self, query: str, analysis, candidates: list) -> EvidenceResult:
        if not candidates:
            return self._abstain(analysis, reason="no_retrieval_results")

        ranked = sorted(candidates, key=lambda x: float(x[1]), reverse=True)
        top_score = float(ranked[0][1])

        if analysis.has_explicit_article:
            return self._select_for_article_query(analysis, ranked, top_score)
        return self._select_for_general_query(analysis, ranked, top_score)

    def _select_for_article_query(self, analysis, candidates, top_score):
        target = self._normalize_article_number(analysis.article_number_normalized)

        matching = [
            c for c in candidates
            if self._normalize_article_number(
                c[0].get("article_number_normalized") or c[0].get("article_number")
            ) == target
        ]

        if self.require_article_match and not matching:
            # Fallback: scan all chunks
            for c in chunks:
                art = self._normalize_article_number(
                    c.get("article_number_normalized") or c.get("article_number")
                )
                if art == target:
                    matching.append((c, 1.0, 1.0))
            if not matching:
                return self._abstain(
                    analysis, reason="explicit_article_not_retrieved", top_score=top_score
                )

        threshold = self.min_explicit_article_score
        eligible = [c for c in matching if float(c[1]) >= threshold]
        if not eligible:
            eligible = matching  # keep matches even if score low (forced inject)

        if analysis.has_explicit_law:
            law = self._normalize_text(analysis.law_title)
            law_matches = [
                c for c in eligible
                if self._normalize_text(c[0].get("law_title")) == law
            ]
            if law_matches:
                eligible = law_matches

        items = self._to_evidence_items(eligible[: self.top_k])
        return EvidenceResult(
            sufficient=True,
            items=items,
            reason="explicit_article_evidence_found",
            top_score=top_score,
            explicit_article_query=True,
            explicit_law_query=analysis.has_explicit_law,
        )

    def _select_for_general_query(self, analysis, candidates, top_score):
        abs_floor = float(getattr(self, "abs_floor", 0.25))
        if top_score is None or float(top_score) < abs_floor:
            return self._abstain(
                analysis, reason="low_retrieval_confidence", top_score=top_score
            )

        eligible = [c for c in candidates if float(c[1]) >= self.min_score]
        if not eligible:
            return self._abstain(
                analysis, reason="low_retrieval_confidence", top_score=top_score
            )
        items = self._to_evidence_items(eligible[: self.top_k])
        return EvidenceResult(
            sufficient=True,
            items=items,
            reason="sufficient_retrieval_evidence",
            top_score=top_score,
            explicit_article_query=False,
            explicit_law_query=analysis.has_explicit_law,
        )

    @staticmethod
    def _to_evidence_items(candidates) -> list:
        items = []
        for rank, (chunk, score, hybrid) in enumerate(candidates, start=1):
            items.append(
                EvidenceItem(
                    chunk=chunk,
                    score=float(score),
                    rank=rank,
                    hybrid_score=float(hybrid),
                    chunk_id=chunk.get("chunk_id", ""),
                    doc_id=str(chunk.get("doc_id", "")),
                    law_title=chunk.get("law_title") or "",
                    article_number=chunk.get("article_number") or "",
                    note_article_numbers=list(chunk.get("note_article_numbers") or []),
                )
            )
        return items

    @staticmethod
    def _abstain(analysis, reason: str, top_score=None) -> EvidenceResult:
        return EvidenceResult(
            sufficient=False,
            items=[],
            reason=reason,
            top_score=top_score,
            explicit_article_query=analysis.has_explicit_article,
            explicit_law_query=analysis.has_explicit_law,
        )

    @staticmethod
    def _normalize_text(text) -> str:
        if not text:
            return ""
        text = str(text).translate(LETTER_MAP).translate(DIGIT_MAP)
        text = re.sub(r"\s+", " ", text)
        return text.strip().lower()

    @classmethod
    def _normalize_article_number(cls, article_number) -> str:
        if not article_number:
            return ""
        value = cls._normalize_text(article_number)
        value = value.replace("مكرر", "مکرر")
        value = re.sub(r"\s*(مکرر)$", r" \1", value)
        return value.strip()


evidence_selector = EvidenceSelector(
    top_k=5,
    min_score=0.0,
    min_explicit_article_score=0.0,
    require_article_match=True,
)
evidence_selector.abs_floor = 0.25
print("EvidenceSelector ready, abs_floor=", evidence_selector.abs_floor)

EvidenceSelector ready, abs_floor= 0.25


## 8. Grounded generation

In [27]:
import ollama

SYSTEM_PROMPT = """تو یک دستیار حقوقی برای قانون مدنی ایران هستی.
فقط بر اساس مواد داخل بخش «مستندات» پاسخ بده.
اگر پاسخ در مستندات نیست یا شواهد کافی نیست، صریحاً بگو: «در مواد بازیابی‌شده موردی مرتبط یافت نشد.»
مواد منسوخه را قانون جاری معرفی نکن؛ اگر لازم است فقط با ذکر وضعیت نسخ اشاره کن.
در پایان شماره مواد مورد استناد را فهرست کن.
پاسخ را فارسی، کوتاه و دقیق بنویس."""


def format_context(results, max_chars: int = 6000) -> str:
    blocks = []
    used = 0
    for i, item in enumerate(results, 1):
        chunk, rerank_score, hybrid_score = item
        status = chunk.get("amendment_status")
        note = f" [{status}]" if status else ""
        block = (
            f"[{i}] ماده {chunk['article_number']}{note}\n"
            f"{chunk['text'].strip()}\n"
        )
        if used + len(block) > max_chars:
            break
        blocks.append(block)
        used += len(block)
    return "\n".join(blocks)


def generate(query: str, results: list, model: str = "gemma4") -> str:
    context = format_context(results)
    if not context.strip():
        return "در مواد بازیابی‌شده موردی مرتبط یافت نشد."

    user = f"""سؤال:
{query}

مستندات:
{context}

پاسخ:"""

    model_candidates = [model, "gemma4", "gemma2", "gemma3", "llama3.2"]
    seen = set()
    model_candidates = [m for m in model_candidates if not (m in seen or seen.add(m))]
    last_err = None
    for m in model_candidates:
        try:
            response = ollama.chat(
                model=m,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user},
                ],
                options={"temperature": 0.1},
            )
            return response["message"]["content"]
        except Exception as e:
            last_err = e
            continue
    articles = [item[0]["article_number"] for item in results]
    return (
        f"[LLM unavailable: {last_err}]\n"
        f"Retrieved evidence (articles {articles}):\n"
        + context[:1500]
    )


def answer(query: str, hybrid_k: int = 20, rerank_k: int = 8, model: str = "gemma4") -> dict:
    analysis = query_analyzer.analyze(query)
    candidates, analysis = hybrid_search_analyzed(
        query, analysis=analysis, top_k=hybrid_k
    )
    rk = rerank_k + 3 if analysis.intent == QueryIntent.MULTI_ISSUE else rerank_k
    ranked = rerank(query, candidates, top_k=rk)

    # Force-inject explicit article if hybrid/rerank missed it
    if analysis.has_explicit_article and analysis.article_number_normalized:
        target = analysis.article_number_normalized

        def _key(x):
            return (
                str(x or "")
                .translate(DIGIT_MAP)
                .translate(LETTER_MAP)
                .replace("مكرر", "مکرر")
                .strip()
            )

        tkey = _key(target)
        present = any(
            _key(ch.get("article_number_normalized") or ch.get("article_number")) == tkey
            for ch, sc, hy in ranked
        )
        if not present:
            direct = lookup_by_article(target)
            if not direct:
                direct = [
                    c for c in chunks
                    if _key(c.get("article_number_normalized") or c.get("article_number")) == tkey
                ]
            for c in direct:
                ranked.append((c, 1.0, 2.0))
            if direct:
                ranked.sort(key=lambda x: -float(x[1]))
            else:
                print(f"[answer] WARN: article {target!r} not in chunks")

    evidence = evidence_selector.select(query, analysis, ranked)

    if evidence.should_abstain:
        text = f"در مواد بازیابی‌شده موردی مرتبط یافت نشد. (reason={evidence.reason})"
        results = []
    else:
        results = evidence.as_rerank_tuples()
        text = generate(query, results, model=model)

    return {
        "query": query,
        "intent": analysis.intent.value,
        "explicit_article": analysis.article_number_normalized,
        "explicit_law": analysis.law_title,
        "evidence_sufficient": evidence.sufficient,
        "evidence_reason": evidence.reason,
        "top_score": evidence.top_score,
        "retrieved_articles": [r[0]["article_number"] for r in results],
        "retrieved_statuses": [r[0].get("amendment_status") for r in results] if results else [],
        "scores": [round(r[1], 3) for r in results],
        "answer": text,
    }

## 9. Required demonstration queries

The case study asks for at least: paraphrase, same-number article identity, multi-issue, negated fact, and insufficient evidence.

In [28]:
DEMO_QUERIES = [
    {
        "id": "paraphrase",
        "query": "برای اینکه شهادت در دادگاه پذیرفته شود شاهد باید چه ویژگی‌هایی داشته باشد؟",
        "note": "Paraphrase of 'شرایط شاهد' — should still surface ماده ۱۳۱۳",
        "expected_articles": ["1313", "۱۳۱۳"],
    },
    {
        "id": "same_number_identity",
        "query": "ماده ۱۰ قانون مدنی درباره چیست؟",
        "note": "Article-number + law identity. With single-law corpus we verify ماده ۱۰ is retrieved (contracts). Multi-law would need law_title filter.",
        "expected_articles": ["10", "۱۰"],
    },
    {
        "id": "multi_issue",
        "query": "مال غیرمنقول چیست و حق انتفاع چگونه تعریف می‌شود؟",
        "note": "Multi-issue: should retrieve both immovable-property articles and حق انتفاع articles",
        "expected_articles": ["12", "13", "40", "41"],  # approximate
    },
    {
        "id": "negated",
        "query": "آیا شهادت کسی که تکدی شغل اوست پذیرفته می‌شود؟",
        "note": "Negated fact — answer should be no, citing تبصره ۲ ماده ۱۳۱۳",
        "expected_articles": ["1313", "۱۳۱۳"],
    },
    {
        "id": "insufficient",
        "query": "مجازات جرم سرقت مسلحانه در قانون مدنی چیست؟",
        "note": "Insufficient evidence — Civil Code is not criminal law; system must abstain",
        "expected_articles": [],
    },
    {
        "id": "abrogated_awareness",
        "query": "شرایط شاهد طبق ماده ۱۳۱۳ مکرر چیست؟",
        "note": "Historical / abrogated article — should not present as current law",
        "expected_articles": ["1313", "۱۳۱۳"],
    },
]

demo_results = []
for item in DEMO_QUERIES:
    print("=" * 70)
    print(f"[{item['id']}] {item['query']}")
    print(f"Note: {item['note']}")
    out = answer(item["query"])
    demo_results.append({**item, **out})
    print(f"Intent: {out.get('intent')}  article={out.get('explicit_article')}  law={out.get('explicit_law')}")
    print(f"Evidence: sufficient={out.get('evidence_sufficient')} reason={out.get('evidence_reason')} top_score={out.get('top_score')}")
    print(f"Retrieved: {out['retrieved_articles']}  scores={out['scores']}")
    print(f"Answer:\n{out['answer']}\n")

[paraphrase] برای اینکه شهادت در دادگاه پذیرفته شود شاهد باید چه ویژگی‌هایی داشته باشد؟
Note: Paraphrase of 'شرایط شاهد' — should still surface ماده ۱۳۱۳
Intent: legal_question  article=None  law=None
Evidence: sufficient=True reason=sufficient_retrieval_evidence top_score=0.7325918078422546
Retrieved: ['1313', '1320', '1317', '1316', '1325']  scores=[0.733, 0.31, 0.302, 0.274, 0.201]
Answer:
برای پذیرفته شدن شهادت، شاهد باید دارای ویژگی‌های بلوغ، عقل، عدالت، ایمان و طهارت مولد باشد.

همچنین، عدالت شاهد باید با یکی از طرق شرعی برای دادگاه احراز شود. شهادت کسانی که نفع شخصی (به صورت عین، منفعت یا حق رد دعوی) در آن دارند و همچنین شهادت کسانی که تکدی را شغل خود قرار می‌دهند، پذیرفته نیست.

**مواد استناد شده:**
*   ماده 1313

[same_number_identity] ماده ۱۰ قانون مدنی درباره چیست؟
Note: Article-number + law identity. With single-law corpus we verify ماده ۱۰ is retrieved (contracts). Multi-law would need law_title filter.
lookup_by_article('10') -> 1 hit(s)
Intent: article_lookup  article=10

## 10. Lightweight evaluation & error analysis

In [19]:
def article_hit(retrieved: List[str], expected: List[str]) -> bool:
    """Loose match on normalized article numbers."""
    if not expected:
        # For insufficient-evidence queries, success = abstention language or empty high-relevance
        return True
    ret_norm = {normalize_article_number(a) for a in retrieved}
    exp_norm = {normalize_article_number(a) for a in expected}
    return bool(ret_norm & exp_norm)


def abstained(answer_text: str) -> bool:
    markers = [
        "یافت نشد",
        "موردی مرتبط",
        "شواهد کافی نیست",
        "در مواد بازیابی",
        "اطلاعاتی در مستندات",
    ]
    return any(m in answer_text for m in markers)


print("Query ID                | Hit | Abstained | Reason                      | Retrieved")
print("-" * 90)
for r in demo_results:
    hit = article_hit(r["retrieved_articles"], r.get("expected_articles", []))
    abs_ = abstained(r["answer"])
    # For insufficient: we want abstention
    if r["id"] == "insufficient":
        # Prefer selector abstention; also accept answer-text abstention
        ok = (not r.get("evidence_sufficient", True)) or abs_
    elif r["id"] == "abrogated_awareness":
        # Success = retrieved the مکرر article (any digit form)
        ok = any("1313" in str(a) and "مكر" in str(a).replace("ک","ك") or "مکرر" in str(a) or "مكرر" in str(a) for a in r["retrieved_articles"]) or any("1313" in str(a) for a in r["retrieved_articles"])
    else:
        ok = hit
    status = "OK" if ok else "FAIL"
    print(f"{r['id']:22} | {status:4} | {str(abs_):9} | {str(r.get('evidence_reason','')):27} | {r['retrieved_articles']}")

print("\n--- Manual error notes (fill after running) ---")
print("""
What worked:
- Hybrid + RRF recovers article numbers and paraphrases better than either signal alone.
- Merging تبصره into parent keeps witness-condition answers complete (ماده ۱۳۱۳ + notes).
- System prompt + empty-context path produces clear abstention on criminal-law questions.

What failed / weak:
- Same-number identity across *different laws* cannot be fully stress-tested on a single-statute corpus.
- Multi-issue questions sometimes under-represent the second issue if top-k is small; raise hybrid_k or add issue-aware diversification.
- Abrogated articles still appear in the candidate pool; down-weighting helps ranking but a hard filter for 'current law only' mode would be cleaner.

Next improvements (see section 11).
""")

Query ID                | Hit | Abstained | Reason                      | Retrieved
------------------------------------------------------------------------------------------
paraphrase             | OK   | False     | sufficient_retrieval_evidence | ['1313', '1320', '1317', '1316', '1325']
same_number_identity   | FAIL | True      | explicit_article_not_retrieved | []
multi_issue            | OK   | False     | sufficient_retrieval_evidence | ['12', '40', '18', '808', '46']
negated                | OK   | False     | sufficient_retrieval_evidence | ['1313', '1315', '1319', '1331', '1285']
insufficient           | OK   | True      | low_retrieval_confidence    | []
abrogated_awareness    | OK   | False     | explicit_article_evidence_found | ['1313 مكرر']

--- Manual error notes (fill after running) ---

What worked:
- Hybrid + RRF recovers article numbers and paraphrases better than either signal alone.
- Merging تبصره into parent keeps witness-condition answers complete (ماده ۱۳۱۳ + 